# Foundations of Data Science
## Tutorial 5 — Principal Component Analysis · **Solutions**

This notebook contains worked solutions to all `# your code here` exercises from `tutorial06_pca.ipynb`. The non-exercise cells are identical so you can run this end-to-end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_wine

rng = np.random.default_rng(0)
plt.rcParams["figure.figsize"] = (9, 4)

## 1. Why dimensionality reduction?

Real datasets often live in very high-dimensional spaces:

- An $8 \times 8$ grayscale image → 64 dimensions
- A $256 \times 256$ photo → 65,536 dimensions
- A document with vocabulary of 50,000 words → 50,000 dimensions

But the *intrinsic* dimensionality is usually much lower. Most natural data sits near a low-dimensional manifold inside the ambient space. PCA finds the best **linear** approximation to that manifold.

## 2. PCA from scratch via SVD

Given $n$ data points stacked as rows of a matrix $X \in \mathbb{R}^{n \times d}$, PCA goes like this:

1. **Center the data:** subtract the column means.
2. **Compute the SVD:** $\tilde X = U \Sigma V^\top$.
3. The columns of $V$ are the **principal components**.
4. $U \Sigma$ holds the **scores**.
5. To reduce to $k$ dimensions, keep only the first $k$ columns of $V$.

In [ ]:
def pca(X, k=None):
    """
    Run PCA on data matrix X (n samples × d features).
    Returns: scores (n × k), components (k × d), singular values, mean.
    """
    mean = X.mean(axis=0)
    X_centered = X - mean
    U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)
    if k is None:
        k = len(s)
    scores      = U[:, :k] * s[:k]
    components  = Vt[:k]
    return scores, components, s, mean

### A 2-D toy example

In [ ]:
n = 200
cov = np.array([[3, 2.5], [2.5, 3]])
X_toy = rng.multivariate_normal(mean=[1, 2], cov=cov, size=n)

scores, components, s, mean = pca(X_toy)
print("Singular values:", s)
print("Principal components (rows):")
print(components)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_toy[:, 0], X_toy[:, 1], alpha=0.5, s=20)
ax.scatter(*mean, color="red", s=80, label="mean")
scale = 0.15
for i in range(2):
    dx, dy = components[i] * s[i] * scale
    ax.arrow(mean[0], mean[1], dx, dy, head_width=0.15, color="red", length_includes_head=True)
    ax.text(mean[0] + dx*1.15, mean[1] + dy*1.15, f"PC{i+1}", color="red", fontsize=12)
ax.set_aspect("equal")
ax.legend()
ax.set_title("PCA finds the axes of greatest variance")
plt.show()

### Exercise 5.0 — verify orthogonality · **Solution**

Compute `components @ components.T`. It should be the identity matrix because the rows of $V^\top$ are orthonormal.

In [ ]:
# Dot product of the two PCs (rows of components)
dot = components[0] @ components[1]
print(f"PC1 · PC2 = {dot:.2e}  (≈ 0, so they are orthogonal)")

# Gram matrix of the components
gram = components @ components.T
print("\ncomponents @ components.T =")
print(np.round(gram, 12))
print("\n→ this is the 2×2 identity, confirming the PCs are orthonormal.")

## 3. PCA on the handwritten digits

In [ ]:
digits = load_digits()
X = digits.data.astype(float)
y = digits.target
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(11, 2))
for i, ax in enumerate(axes):
    idx = np.where(y == i)[0][0]
    ax.imshow(X[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title(str(i))
    ax.axis("off")
plt.suptitle("One example of each digit", y=1.05)
plt.show()

In [ ]:
scores, components, s, mean = pca(X)
print(f"scores shape:     {scores.shape}")
print(f"components shape: {components.shape}")
print(f"singular values:  {s[:5]} ...")

## 4. Visualize the digits in 2-D

In [ ]:
plt.figure(figsize=(8, 6))
sc = plt.scatter(scores[:, 0], scores[:, 1], c=y, cmap="tab10", s=15, alpha=0.8)
plt.colorbar(sc, label="digit", ticks=range(10))
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("Digits projected onto the first two principal components")
plt.show()

### Exercise 5.1 — scatter plots in other PC pairs · **Solution**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

pc_pairs = [(0, 1), (1, 2), (0, 4)]   # (PC1, PC2), (PC2, PC3), (PC1, PC5) — 0-indexed
labels   = ["PC1 vs PC2", "PC2 vs PC3", "PC1 vs PC5"]

for ax, (i, j), label in zip(axes, pc_pairs, labels):
    sc = ax.scatter(scores[:, i], scores[:, j], c=y, cmap="tab10", s=12, alpha=0.8)
    ax.set_xlabel(f"PC {i+1}")
    ax.set_ylabel(f"PC {j+1}")
    ax.set_title(label)

fig.colorbar(sc, ax=axes, label="digit", ticks=range(10), shrink=0.8)
plt.show()

print("Observation: PC1/PC2 typically separates the clusters best because those")
print("directions capture the most variance. Higher PCs encode finer structure")
print("that is less aligned with the digit classes — clusters overlap more.")

## 5. What do the principal components look like?

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(components[i].reshape(8, 8), cmap="RdBu", vmin=-0.4, vmax=0.4)
    ax.set_title(f"PC {i+1}\nσ = {s[i]:.0f}")
    ax.axis("off")
plt.suptitle("The first 10 principal components, reshaped as images", y=1.02)
plt.tight_layout()
plt.show()

## 6. Scree plot — choosing the number of components

In [ ]:
explained = s**2 / np.sum(s**2)
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(s)+1), explained, color="steelblue")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Fraction of variance explained")
axes[0].set_title("Scree plot")
axes[0].set_xlim(0, 30)

axes[1].plot(range(1, len(s)+1), cumulative, marker="o", ms=4, color="darkorange")
axes[1].axhline(0.9,  color="gray", linestyle="--", label="90%")
axes[1].axhline(0.95, color="gray", linestyle=":",  label="95%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title("Cumulative explained variance")
axes[1].legend()
axes[1].set_xlim(0, 64)
axes[1].set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

In [ ]:
for threshold in [0.5, 0.8, 0.9, 0.95, 0.99]:
    k_needed = np.argmax(cumulative >= threshold) + 1
    print(f"{int(threshold*100)}% of variance needs {k_needed:2d} components")

### Exercise 5.2 — compression at k=8 · **Solution**

In [ ]:
k = 8
variance_kept = cumulative[k - 1]              # cumulative is 0-indexed
variance_lost = 1 - variance_kept
compression   = X.shape[1] / k                 # 64 / 8 = 8x

print(f"With k = {k} components:")
print(f"  Variance kept     : {variance_kept*100:.2f}%")
print(f"  Variance lost     : {variance_lost*100:.2f}%")
print(f"  Compression ratio : {compression:.1f}x  ({X.shape[1]} → {k} numbers per image)")

## 7. Reconstruction at low rank

In [ ]:
def reconstruct(X, k):
    scores, components, s, mean = pca(X, k=k)
    return scores @ components + mean

ks = [1, 2, 5, 10, 20, 64]
sample_idx = [0, 100, 300, 700, 1400]

fig, axes = plt.subplots(len(sample_idx), len(ks) + 1, figsize=(11, 7))
for row, idx in enumerate(sample_idx):
    axes[row, 0].imshow(X[idx].reshape(8, 8), cmap="gray_r")
    axes[row, 0].set_ylabel(f"digit {y[idx]}")
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    if row == 0:
        axes[row, 0].set_title("original")

    for col, k in enumerate(ks, start=1):
        X_hat = reconstruct(X, k)
        axes[row, col].imshow(X_hat[idx].reshape(8, 8), cmap="gray_r")
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if row == 0:
            axes[row, col].set_title(f"k = {k}")

plt.suptitle("Reconstruction quality vs. number of components", y=1.0)
plt.tight_layout()
plt.show()

## Sanity check: scikit-learn's PCA

In [ ]:
from sklearn.decomposition import PCA

pca_sk = PCA(n_components=10)
scores_sk = pca_sk.fit_transform(X)

print("Our PC1 std :", scores[:, 0].std())
print("sk PC1 std  :", scores_sk[:, 0].std())
print("Our explained var ratio (top 5):", (s**2/(s**2).sum())[:5])
print("sk  explained var ratio (top 5):", pca_sk.explained_variance_ratio_[:5])

## 8. Exercise 5.3 — PCA on the wine dataset · **Solution**

In [ ]:
wine = load_wine()
X_wine = wine.data
y_wine = wine.target
print("feature names:", wine.feature_names)
print("\nfeature stds (raw):")
for name, std in zip(wine.feature_names, X_wine.std(axis=0)):
    print(f"  {name:25s} {std:8.3f}")

### Part 1: Why standardize?

Look at the stds above. `proline` has std ≈ 314, while `nonflavanoid_phenols` has std ≈ 0.12 — that's a factor of about **2,600**. PCA finds directions of **greatest variance**, so it will be utterly dominated by the high-variance features unless we put them all on the same scale. Standardization (subtract mean, divide by std per feature) fixes this.

In [ ]:
# Part 2: standardize, then PCA
X_std = (X_wine - X_wine.mean(axis=0)) / X_wine.std(axis=0)
scores_w, comps_w, s_w, mean_w = pca(X_std)

explained_w = s_w**2 / np.sum(s_w**2)
cumulative_w = np.cumsum(explained_w)
print("Explained variance ratio (first 5 PCs):", np.round(explained_w[:5], 4))

In [ ]:
# Part 3: 2-D scatter colored by class
plt.figure(figsize=(8, 6))
for cls in np.unique(y_wine):
    mask = y_wine == cls
    plt.scatter(scores_w[mask, 0], scores_w[mask, 1], label=f"class {cls}", s=40, alpha=0.8)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("Wine cultivars in PCA space (standardized features)")
plt.legend()
plt.show()

print("The three cultivars separate nicely along PC1 and PC2.")

In [ ]:
# Part 4: scree plot — how many PCs for 90%?
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(s_w)+1), explained_w, color="steelblue")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Fraction of variance explained")
axes[0].set_title("Scree plot — wine (standardized)")

axes[1].plot(range(1, len(s_w)+1), cumulative_w, marker="o", color="darkorange")
axes[1].axhline(0.9, color="gray", linestyle="--", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title("Cumulative explained variance")
axes[1].legend()
plt.tight_layout()
plt.show()

k90 = np.argmax(cumulative_w >= 0.9) + 1
print(f"Need k = {k90} components to keep 90% of variance.")

In [ ]:
# Part 5: same thing without standardization
scores_raw, comps_raw, s_raw, _ = pca(X_wine)   # mean-centered only
explained_raw = s_raw**2 / np.sum(s_raw**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cls in np.unique(y_wine):
    mask = y_wine == cls
    axes[0].scatter(scores_raw[mask, 0], scores_raw[mask, 1], label=f"class {cls}", s=40, alpha=0.8)
axes[0].set_title("Unstandardized PCA")
axes[0].set_xlabel("PC 1"); axes[0].set_ylabel("PC 2")
axes[0].legend()

for cls in np.unique(y_wine):
    mask = y_wine == cls
    axes[1].scatter(scores_w[mask, 0], scores_w[mask, 1], label=f"class {cls}", s=40, alpha=0.8)
axes[1].set_title("Standardized PCA")
axes[1].set_xlabel("PC 1"); axes[1].set_ylabel("PC 2")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Unstandardized: PC1 explains {explained_raw[0]*100:.1f}% of variance")
print(f"Standardized:   PC1 explains {explained_w[0]*100:.1f}% of variance\n")
print("Without standardization, PC1 is essentially just the proline feature")
print("(the highest-variance one in raw units). All other chemical features")
print("are ignored, and the cultivars don't separate as cleanly.")

# Show how aligned PC1 is with the proline axis (feature index 12)
proline_loading = np.abs(comps_raw[0, 12])
print(f"\n|PC1's loading on proline| = {proline_loading:.4f}  (close to 1 means almost identical)")

## Takeaways

- **PCA = SVD on centered data.** No new math, just a recipe.
- The principal components are the columns of $V$; the scores are $U\Sigma$.
- Use a **scree plot** to pick $k$. Common thresholds: 90% or 95% of variance.
- The rank-$k$ truncation gives the optimal rank-$k$ approximation (Eckart-Young).
- **Always standardize** when features have different units — otherwise high-variance features dominate.
- PCA is **linear** — for highly non-linear structure use t-SNE, UMAP, or autoencoders.